In [ ]:
import numpy as np
import torch
from matplotlib import pyplot as plt
import torcwa
from tqdm.notebook import tqdm
from pvlib import spectrum
from refractiveindex import RefractiveIndexMaterial
import os
%load_ext line_profiler

# Hardware
# If GPU support TF32 tensor core, the matmul operation is faster than FP32 but with less precision.
# If you need accurate operation, you have to disable the flag below.
torch.backends.cuda.matmul.allow_tf32 = False
sim_dtype = torch.complex64
geo_dtype = torch.float32
device = torch.device('cuda')

def get_sine_eps(x,params,grating_period,eps):
    """Generate sine grating permittivity profile.

    Args:
        x (torch.tensor): 1D tensor of x positions.
        params (torch.Tensor): list of amplitude and phase shift. shape (n,2), where n is n*2*np.pi/grating_period'th frequency.
        eps (float): Permittivity of high-index material.

    Returns:
        torch.tensor: 1D tensor of permittivity profile.
    """
    A = torch.sum(params[:,0]) + 1e-9
    cosines = torch.cos(2.*np.pi*torch.arange(1, params.shape[0]+1, 
                                              dtype=geo_dtype,device=device).unsqueeze(1)*(x.unsqueeze(0)/grating_period)
                                               - params[:,1].unsqueeze(1))
    cosines = cosines * params[:,0].unsqueeze(1)
    eps = 1 + (eps-1)*(0.5*(A+torch.sum(cosines, dim=0))/A)
    return eps.unsqueeze(1)   # make shape (nx,1) so add_layer accepts it

# Simulation environment
# light
spectra = spectrum.get_reference_spectra()
am15g = spectra['global']
wavelengths = torch.linspace(300,1100,100,dtype=int) # nm
sun_weights = lambda x: torch.tensor(am15g[x.cpu().numpy()])

# material
si = RefractiveIndexMaterial(shelf='main', book='Si', page='Green-2008')
si_eps = lambda x: torch.tensor(si.get_refractive_index(x) +
                      1j * si.get_extinction_coefficient(x))**2

# geometry
h = 1000 #nm
grating_period = 1000 # nm
L = [grating_period, 1.]  # nm
torcwa.rcwa_geo.dtype = geo_dtype
torcwa.rcwa_geo.device = device
torcwa.rcwa_geo.Lx = L[0]
torcwa.rcwa_geo.Ly = L[1]
torcwa.rcwa_geo.ny = 1

Order_N = 40
order = [Order_N,0]

In [19]:
def get_absorptance(params,wavelength,inc_ang,azi_ang):
    # calculate incidence power from inc_ang, azi_ang
    P_inc = 0.5 / np.cos(inc_ang) * (np.cos(azi_ang)**2 + np.sin(azi_ang)**2*np.cos(inc_ang)**2)
    #See Silicon Modulation Sanity Checks.ipynb for more details
    P_inc = P_inc*torcwa.rcwa_geo.Lx

    #setup simulation
    torcwa.rcwa_geo.nx = 5000 #set sine grating grid
    torcwa.rcwa_geo.grid()
    A = torch.sum(params[:,0])
    sine_eps = get_sine_eps(torcwa.rcwa_geo.x,params=params,grating_period=grating_period,eps=si_eps(wavelength))
    sim = torcwa.rcwa(freq=1/wavelength,order=order,L=L,dtype=sim_dtype,device=device)
    sim.add_input_layer()
    sim.add_output_layer()
    sim.set_incident_angle(inc_ang=inc_ang,azi_ang=azi_ang)
    sim.add_layer(thickness=A,eps=sine_eps)
    sim.add_layer(thickness=h,eps=si_eps(wavelength))
    sim.solve_global_smatrix()

    # choose probe planes (just above / below film). use same device dtype as sim
    z_top = torch.clone(A)  # e.g. top of film
    z_bot = torch.clone(A+h)  # e.g. bottom of film 
    z_air = torch.tensor(-3*h,device=sim._device, dtype=geo_dtype) #Ensure evanescent modes are gone
    torcwa.rcwa_geo.nx = 5000 #set sampling grid for fields
    torcwa.rcwa_geo.grid()
    P_absorbed_film = torch.zeros(2,device=sim._device, dtype=geo_dtype) # to store absorbed power in film, first item is x polarization, second y polarization
    P_absorbed_grating = torch.zeros(2,device=sim._device, dtype=geo_dtype) # to store absorbed power in grating, first item is x polarization, second y polarization
    A_film = torch.zeros(2,device=sim._device, dtype=geo_dtype) #to store absorptance in film. first item is x polarization, second y polarization
    A_grating = torch.zeros(2,device=sim._device, dtype=geo_dtype) #to store absorptance in grating. first item is x polarization, second y polarization
    P_slices = torch.zeros((2,3),device=sim._device, dtype=geo_dtype) # to store Poynting vector slices at each plane for both polarizations
    Reflectance = torch.zeros(2,device=sim._device, dtype=geo_dtype) # to store reflectance for both polarizations first item is x polarization, second y polarization
    Transmittance = torch.zeros(2,device=sim._device, dtype=geo_dtype) # to store transmittance for both polarizations first item is x polarization, second y polarization

    sim.source_planewave(amplitude=[1.,0.],direction='forward',notation='xy')
    # request fields at both planes: x_axis is your x sampling (1D tensor), y0 is y coordinate (often 0)
    [Ex, Ey, Ez], [Hx, Hy, Hz] = sim.field_xz(torcwa.rcwa_geo.x, torch.stack((z_top,z_bot,z_air)), y=0.0)
    # Ex,Hy shapes: (nx, 2)  (nx across x, 2 planes)
    S_z = 0.5 * torch.real(Ex * torch.conj(Hy) - Ey * torch.conj(Hx))   # shape (nx,3)
    P_top = torch.trapezoid(S_z[:,0], torcwa.rcwa_geo.x)
    P_bot = torch.trapezoid(S_z[:,1], torcwa.rcwa_geo.x)
    P_air = torch.trapezoid(S_z[:,2], torcwa.rcwa_geo.x)
    P_absorbed_film[0] = P_top - P_bot
    P_absorbed_grating[0] = P_air - P_top
    A_film[0] = P_absorbed_film[0] / P_inc
    A_grating[0] = P_absorbed_grating[0] / P_inc
    P_slices[0,:] = torch.tensor([P_top, P_bot, P_air], device=sim._device, dtype=geo_dtype)
    Reflectance[0]  = (P_inc - P_air) / P_inc
    Transmittance[0] = P_bot / P_inc

    sim.source_planewave(amplitude=[0.,1.],direction='forward',notation='xy')
    # request fields at both planes: x_axis is your x sampling (1D tensor), y0 is y coordinate (often 0)
    [Ex, Ey, Ez], [Hx, Hy, Hz] = sim.field_xz(torcwa.rcwa_geo.x, torch.stack((z_top,z_bot,z_air)), y=0.0)
    # Ex,Hy shapes: (nx, 2)  (nx across x, 2 planes)
    S_z = 0.5 * torch.real(Ex * torch.conj(Hy) - Ey * torch.conj(Hx))   # shape (nx,3)
    P_top = torch.trapezoid(S_z[:,0], torcwa.rcwa_geo.x)
    P_bot = torch.trapezoid(S_z[:,1], torcwa.rcwa_geo.x)
    P_air = torch.trapezoid(S_z[:,2], torcwa.rcwa_geo.x)
    P_absorbed_film[1] = P_top - P_bot
    P_absorbed_grating[1] = P_air - P_top
    A_film[1] = P_absorbed_film[1] / P_inc
    A_grating[1] = P_absorbed_grating[1] / P_inc
    P_slices[1,:] = torch.tensor([P_top, P_bot, P_air], device=sim._device, dtype=geo_dtype)
    Reflectance[1]  = (P_inc - P_air) / P_inc
    Transmittance[1] = P_bot / P_inc

    return  A_film, A_grating, Reflectance, Transmittance, P_absorbed_film, P_absorbed_grating, P_slices

In [ ]:
n_structures = 5000
n_wavelengths = len(wavelengths)
sum_am15g = torch.sum(torch.tensor([sun_weights(wavelength) for wavelength in wavelengths], device=device))
sum_photons = torch.sum(torch.tensor([sun_weights(wavelength)*wavelength for wavelength in wavelengths], device=device)) 

# 1. Pre-allocate dense tensors
# Shape: (Structure, Wavelength, Polarization) or similar
all_params = torch.zeros((n_structures, 5, 2), dtype=geo_dtype)
all_A_film = torch.zeros((n_structures, n_wavelengths, 2), dtype=geo_dtype)
all_A_grat = torch.zeros((n_structures, n_wavelengths, 2), dtype=geo_dtype)
all_Refl   = torch.zeros((n_structures, n_wavelengths, 2), dtype=geo_dtype)
all_Trans  = torch.zeros((n_structures, n_wavelengths, 2), dtype=geo_dtype)
all_targets = torch.zeros((n_structures, 2), dtype=torch.float32)

with tqdm(total=n_structures * n_wavelengths, desc="Total sims", unit="run") as pbar:
    for i in range(n_structures):
        # Generate params once per structure
        params = torch.zeros((5,2), dtype=geo_dtype, device=device)
        params[:,0] = torch.rand((5,), device=device) * 40.0 # amplitude (nm)
        params[:,1] = torch.rand((5,), device=device) * 2. * np.pi # phase (rad)
        
        # Store params once
        all_params[i] = params.cpu()

        running_sun_weight = 0.0
        running_photon_weight = 0.0

        for j, wavelength in enumerate(wavelengths):
            # Run sim
            A_f, A_g, R, T, _, _, _ = get_absorptance(params, wavelength, 0, 0)
            
            # Store results directly into the dense tensor by index
            all_A_film[i, j] = A_f.cpu()
            all_A_grat[i, j] = A_g.cpu()
            all_Refl[i, j]   = R.cpu()
            all_Trans[i, j]  = T.cpu()
            
            # Accumulate targets
            running_sun_weight += torch.mean(A_f) * sun_weights(wavelength)
            running_photon_weight += torch.mean(A_f) * sun_weights(wavelength) * wavelength
            pbar.update(1)

        # Calculate and store targets
        all_targets[i, 0] = (running_sun_weight / sum_am15g).cpu()
        all_targets[i, 1] = (running_photon_weight / sum_photons).cpu()

        # Save periodically
        if (i + 1) % 1000 == 0:
            combined_data = {
                "features": all_params[:i+1],
                "targets": all_targets[:i+1],
                "A_film": all_A_film[:i+1],
                "A_grating": all_A_grat[:i+1],
                "Reflectance": all_Refl[:i+1],
                "Transmittance": all_Trans[:i+1],
                "Wavelengths": wavelengths
            }
            torch.save(combined_data, f'../Data/Modulated_Silicon_Data_{i+1}_structures.pt')